<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Game_Ver2/cournot_10d_two_block_mlp_deterministic_dtb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10D two-block Cournot benchmark with an ordinary MLP: deterministic DTB and explicit Euler

This notebook applies the same deterministic MLP-DTB setup as the five-player comparison to the [10D two-block Cournot-type benchmark](https://chatgpt.com/share/6aa7080e-8ee4-83ea-bf29-8eefb509c5c0). The evolving MLP supplies the tangent field; particle values are accumulated separately and are never replaced by a neural-map evaluation. There are no resets or NN refits.

Split the coordinates into blocks $A=\{1,\ldots,5\}$ and $B=\{6,\ldots,10\}$. For each player, $s_i$ is the sum of the other four coordinates in the same block. With

$$R_\mu(s)=\max\{\mu s(1-s),0\},\qquad b=2,$$

use

$$\dot X_i=2b\bigl(R_{\mu_A}(s_i)-X_i\bigr),\quad i\in A,
\qquad \mu_A=\frac74,$$

and

$$\dot X_i=2b\bigl(R_{\mu_B}(s_i)-X_i\bigr),\quad i\in B,
\qquad \mu_B=\frac{33}{20}.$$

The uncoupled benchmark has 50 stable product equilibria: block $A$ has three active coordinates equal to $5/14$, while block $B$ has four active coordinates equal to $79/297$. Its local spectrum includes fast modes down to $-15.8$ and three deliberately slow modes equal to $-1/15$.

DTB projects $F(X_k)$ onto the current neural tangent basis. The reference advances the same labeled initial particles with explicit Euler and the exact same time grid. The global experiment starts from $U([0,1]^{10})$, as specified for the benchmark. There is no stochastic score correction or dynamical noise.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Use the existing helpers locally, or obtain them when running in Colab.
module_files = ('dtb.py', 'run_game_dtb.py')
module_dir = next((p for p in (Path.cwd(), Path.cwd() / 'DTB_Game_Ver2')
                   if all((p / name).is_file() for name in module_files)), None)
if module_dir is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run from the repository root or DTB_Game_Ver2.')
    repo = Path('/content/dtb-colab-experiments')
    if not repo.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo),
        ], check=True)
    module_dir = repo / 'DTB_Game_Ver2'
sys.path.insert(0, str(module_dir.resolve()))

from dtb import flat_params
from run_game_dtb import ResidualMLPMap, game_dtb_basis_matrices

output_root = Path('/content') if Path('/content').is_dir() else module_dir
output_dir = output_root / 'cournot_10d_two_block_mlp_deterministic_results'
output_dir.mkdir(parents=True, exist_ok=True)

## Parameters

Apart from the 10D vector field and its exact-uniform initial law, the numerical setup matches the five-player deterministic MLP notebook: seed, sample count, time grid, 64-coordinate tangent solve, SVD cutoff, two-layer width-16 tanh MLP, device selection, and float64 arithmetic.

In [ ]:
SEED = 0
DIM = 10
BLOCK_SIZE = 5
COURNOT_B = 2.0
COURNOT_MU_A = 7 / 4
COURNOT_MU_B = 33 / 20
N_PARTICLES = 3000

INITIAL_LAW = 'uniform'
SMOOTHING_STD = 0.02

H = 0.005
T_FINAL = 2
SNAPSHOT_TIMES = (0.0, 0.5, 1.0, T_FINAL)
WIDTH, DEPTH = 16, 2
BASIS_SIZE = 64
SVD_RTOL = 1e-6  # Retain singular values above SVD_RTOL * s_max.
JACOBIAN_CHUNK = 256

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float64
torch.set_num_threads(min(4, torch.get_num_threads()))

if DIM != 2 * BLOCK_SIZE:
    raise ValueError('DIM must contain two blocks of BLOCK_SIZE coordinates.')
if INITIAL_LAW not in ('uniform', 'smoothed_uniform'):
    raise ValueError("INITIAL_LAW must be 'uniform' or 'smoothed_uniform'.")
if SMOOTHING_STD <= 0 or min(H, T_FINAL) <= 0:
    raise ValueError('SMOOTHING_STD, H, and T_FINAL must be positive.')


def game_velocity(x):
    x_a = x[..., :BLOCK_SIZE]
    x_b = x[..., BLOCK_SIZE:]
    rivals_a = x_a.sum(dim=-1, keepdim=True) - x_a
    rivals_b = x_b.sum(dim=-1, keepdim=True) - x_b
    response_a = (COURNOT_MU_A * rivals_a * (1 - rivals_a)).clamp_min(0.0)
    response_b = (COURNOT_MU_B * rivals_b * (1 - rivals_b)).clamp_min(0.0)
    velocity_a = 2 * COURNOT_B * (response_a - x_a)
    velocity_b = 2 * COURNOT_B * (response_b - x_b)
    return torch.cat((velocity_a, velocity_b), dim=-1)

## Evolve parameters and particle values

At step $k$, choose a 64-coordinate parameter subset and evaluate the neural tangent basis at the current physical particles:

$$J_i^k=\partial_{\theta_{S_k}}f_{\theta_k}(x_i^k),\qquad g_i^k=F(x_i^k).$$

The truncated-SVD solve projects $g^k$ onto $J^k$. With $u_i^k=J_i^k\alpha^k$, update

$$\theta_{k+1}=\theta_k+h_kE_{S_k}\alpha^k,
\qquad x_i^{k+1}=x_i^k+h_ku_i^k.$$

The new particle values are not obtained from $f_{\theta_{k+1}}$. The NN parameters only produce the next tangent field. `X_final` stores the final accumulated particles, and the projection error compares $J^k\alpha^k$ with $F(x^k)$.

In [ ]:
# Rerunning this cell restarts DTB; the later Euler reference starts from the same z.
torch.manual_seed(SEED)
initial_generator = torch.Generator().manual_seed(SEED)
z_cpu = torch.rand((N_PARTICLES, DIM), dtype=DTYPE, generator=initial_generator)
if INITIAL_LAW == 'smoothed_uniform':
    z_cpu = z_cpu + SMOOTHING_STD * torch.randn(
        (N_PARTICLES, DIM), dtype=DTYPE, generator=initial_generator
    )
z = z_cpu.to(DEVICE)
x = z.clone()

model = ResidualMLPMap(
    dim=DIM, width=WIDTH, depth=DEPTH, activation='tanh', dtype=DTYPE,
    zero_init_output=False,
).net.to(DEVICE)
theta, structure, _ = flat_params(model)
basis_generator = torch.Generator().manual_seed(SEED + 1)

snapshot_times = np.unique(np.round(SNAPSHOT_TIMES, 12))
times = np.unique(np.round(np.r_[np.arange(0, T_FINAL, H), snapshot_times, T_FINAL], 12))
projection_times = times[:-1]

projection_error = []
jacobian_sigma_max = []
jacobian_sigma_min = []
jacobian_condition = []
clouds = {0.0: x.cpu().numpy().copy()}

for k, t in enumerate(projection_times):
    target_velocity = game_velocity(x)

    indices = torch.randperm(theta.numel(), generator=basis_generator)
    selected = indices[:BASIS_SIZE].sort().values.to(DEVICE)
    theta_step = theta.detach().clone()
    _, _, J_flat = game_dtb_basis_matrices(
        theta_step, selected, x, model, structure, chunk=JACOBIAN_CHUNK)
    J_flat = J_flat.detach()

    A = J_flat / np.sqrt(N_PARTICLES)
    target = target_velocity.reshape(-1) / np.sqrt(N_PARTICLES)
    U, s, Vh = torch.linalg.svd(A, full_matrices=False)
    sigma_max, sigma_min = s[0].item(), s[-1].item()
    jacobian_sigma_max.append(sigma_max * np.sqrt(N_PARTICLES))
    jacobian_sigma_min.append(sigma_min * np.sqrt(N_PARTICLES))
    jacobian_condition.append(sigma_max / sigma_min if sigma_min > 0 else np.inf)
    keep = s > SVD_RTOL * s[0]
    if not keep.any():
        raise FloatingPointError(f'No tangent singular value retained at t={t:g}.')
    alpha = Vh[keep].T @ ((U[:, keep].T @ target) / s[keep])

    tangent_velocity = (J_flat @ alpha).reshape_as(x)
    error = (tangent_velocity - target_velocity).square().sum(dim=1).mean().sqrt()
    if not torch.isfinite(error):
        raise FloatingPointError(f'Nonfinite tangent projection at t={t:g}.')
    projection_error.append(error.item())

    next_time = float(times[k + 1])
    h = next_time - float(t)
    x = (x + h * tangent_velocity).detach()
    theta = theta_step.clone()
    theta[selected] += h * alpha
    if not torch.isfinite(theta).all() or not torch.isfinite(x).all():
        raise FloatingPointError(f'Nonfinite state at t={next_time:g}.')

    if next_time in snapshot_times:
        clouds[next_time] = x.cpu().numpy().copy()

X_final = x.detach().clone()
projection_error = np.asarray(projection_error)
jacobian_sigma_max = np.asarray(jacobian_sigma_max)
jacobian_sigma_min = np.asarray(jacobian_sigma_min)
jacobian_condition = np.asarray(jacobian_condition)

np.save(output_dir / 'dtb_final_particles.npy', X_final.cpu().numpy())
np.savetxt(
    output_dir / 'deterministic_dtb_diagnostics.csv',
    np.column_stack((projection_times, projection_error)),
    delimiter=',', header='time,projection_error', comments='',
)
print(f'Deterministic DTB complete: {N_PARTICLES} particles, '
      f'{len(projection_times)} steps, final time={times[-1]:g}.')

## Deterministic DTB point-cloud projections

Rows show all nine adjacent coordinate pairs $(x_1,x_2)$ through $(x_9,x_{10})$; columns show physical time. The $(x_5,x_6)$ row crosses the two blocks. These are the independently accumulated DTB particles, shown with common axis limits.

In [ ]:
pairs = tuple((i, i + 1) for i in range(DIM - 1))
fig, axes = plt.subplots(
    len(pairs), len(clouds), figsize=(3.5 * len(clouds), 2.7 * len(pairs)),
    sharex=True, sharey=True, squeeze=False, layout='constrained',
)
lo = min(points.min() for points in clouds.values())
hi = max(points.max() for points in clouds.values())
padding = 0.04 * max(hi - lo, 1e-6)

for row, (i, j) in enumerate(pairs):
    for col, (t, points) in enumerate(clouds.items()):
        ax = axes[row, col]
        ax.scatter(points[:, i], points[:, j], s=3, alpha=0.35,
                   color='#176b87', linewidths=0, rasterized=True)
        ax.set(xlabel=fr'$x_{i + 1}$', ylabel=fr'$x_{j + 1}$',
               xlim=(lo - padding, hi + padding), ylim=(lo - padding, hi + padding))
        ax.set_aspect('equal', adjustable='box')
        if row == 0:
            ax.set_title(f't = {t:g}')
fig.suptitle(
    fr'10D two-block deterministic DTB: $b={COURNOT_B:g}$, '
    fr'$\mu_A={COURNOT_MU_A:g}$, $\mu_B={COURNOT_MU_B:g}$'
)
fig.savefig(output_dir / 'point_clouds.png', dpi=300, bbox_inches='tight')
plt.show()

## Projection-error trajectory

The residual is measured against the deterministic 10D target $F(X_k)$:

$$\left(\frac1N\sum_{i=1}^N\|J_i^k\alpha^k-F(x_i^k)\|_2^2\right)^{1/2}.$$

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5), layout='constrained')
ax.plot(projection_times, projection_error, color='#176b87', linewidth=1.5)
ax.set(xlabel='Time', ylabel='RMS projection residual',
       title='Deterministic tangent-projection error',
       xlim=(0, T_FINAL), ylim=(0, None))
ax.grid(alpha=0.2)
fig.savefig(output_dir / 'projection_error.png', dpi=300, bbox_inches='tight')
plt.show()

## Jacobian condition-number trajectory

At the start of each DTB step, record

$$\kappa_2(J^k)=\frac{\sigma_{\max}(J^k)}{\sigma_{\min}(J^k)}.$$

`J_flat` has shape `(N_PARTICLES * DIM, BASIS_SIZE)` for that step's selected parameter coordinates. The same SVD used by the deterministic projection supplies this diagnostic before `SVD_RTOL` filtering. Infinite values mean the computed minimum singular value is exactly zero.

In [ ]:
# Run after the DTB evolution cell to use its recorded singular values.
condition_finite = np.isfinite(jacobian_condition)
condition_infinite = np.isinf(jacobian_condition)
condition_fig, condition_ax = plt.subplots(figsize=(8, 4), layout='constrained')
condition_ax.plot(
    projection_times,
    np.where(condition_finite, jacobian_condition, np.nan),
    color='#7c3aed', linewidth=1.2,
    label=r'$\sigma_{\max}/\sigma_{\min}$',
)
if not condition_finite.any():
    condition_ax.set_ylim(1, 10)
condition_ax.set_yscale('log')
if condition_infinite.any():
    condition_ax.scatter(
        projection_times[condition_infinite],
        np.full(condition_infinite.sum(), 0.96),
        transform=condition_ax.get_xaxis_transform(),
        color='#dc2626', marker='x', s=20,
        label=r'$\infty$ (zero minimum; marked at top)', zorder=3,
    )
condition_ax.set(
    xlabel='Time', ylabel=r'Condition number $\kappa_2(J)$ (log scale)',
    title='Selected Jacobian condition number', xlim=(0, T_FINAL),
)
condition_ax.grid(alpha=0.2, which='both')
condition_ax.legend(loc='best', fontsize=9)
condition_fig.savefig(
    output_dir / 'jacobian_condition_number.png', dpi=300, bbox_inches='tight',
)
np.savetxt(
    output_dir / 'jacobian_condition_number.csv',
    np.column_stack((projection_times, jacobian_sigma_max,
                     jacobian_sigma_min, jacobian_condition)),
    delimiter=',', header='time,sigma_max,sigma_min,condition_number', comments='',
)
print(f'Recorded {len(jacobian_condition)} Jacobian condition numbers; '
      f'{condition_infinite.sum()} infinite.')
plt.show()


## Explicit Euler reference on the same initial cloud and time grid

The deterministic reference uses the same labeled initial particles, every step size, and every snapshot time as DTB:

$$X_{k+1}^{\mathrm{Euler}}=X_k^{\mathrm{Euler}}+h_kF(X_k^{\mathrm{Euler}}).$$

There is no sampled noise and no score correction. Since both methods start from identical labeled particles, the paired-particle RMS is a meaningful trajectory error.

In [ ]:
# Run after the DTB evolution cell; rerunning starts from the same z.
x_euler = z.detach().clone()
clouds_euler = {float(times[0]): x_euler.cpu().numpy().copy()}
euler_snapshot_times = set(float(value) for value in snapshot_times)

with torch.no_grad():
    for euler_t, euler_next_time in zip(times[:-1], times[1:]):
        euler_h = float(euler_next_time - euler_t)
        x_euler = x_euler + euler_h * game_velocity(x_euler)
        if not torch.isfinite(x_euler).all():
            raise FloatingPointError(f'Nonfinite Euler state at t={euler_next_time:g}.')
        if float(euler_next_time) in euler_snapshot_times:
            clouds_euler[float(euler_next_time)] = x_euler.cpu().numpy().copy()

X_euler_final = x_euler.detach().clone()
euler_final_rms_distance = (
    (X_final - X_euler_final).square().sum(dim=1).mean().sqrt().item()
)
euler_final_mean_distance = torch.linalg.vector_norm(
    X_final.mean(dim=0) - X_euler_final.mean(dim=0)
).item()
np.save(output_dir / 'euler_final_particles.npy', X_euler_final.cpu().numpy())
print(f'Explicit Euler reference: {z.shape[0]} particles, {len(times) - 1} steps, '
      f'nominal H={H:g}, final time={times[-1]:g}.')
print(f'Final sample-mean distance (DTB vs Euler): {euler_final_mean_distance:.6g}')
print(f'Final paired-particle RMS (DTB vs Euler): {euler_final_rms_distance:.6g}')

## Explicit Euler reference: all adjacent projection planes

Rows show $(x_1,x_2)$ through $(x_9,x_{10})$; columns show the same snapshot times as the DTB figure. The $(x_5,x_6)$ row crosses the two blocks. Common axis limits cover both methods.

In [ ]:
euler_pairs = tuple((i, i + 1) for i in range(DIM - 1))
euler_fig, euler_axes = plt.subplots(
    len(euler_pairs), len(clouds_euler),
    figsize=(3.5 * len(clouds_euler), 2.7 * len(euler_pairs)),
    sharex=True, sharey=True, squeeze=False, layout='constrained',
)
euler_all_clouds = list(clouds.values()) + list(clouds_euler.values())
euler_lo = min(points.min() for points in euler_all_clouds)
euler_hi = max(points.max() for points in euler_all_clouds)
euler_padding = 0.04 * max(euler_hi - euler_lo, 1e-6)

for euler_row, (euler_i, euler_j) in enumerate(euler_pairs):
    for euler_col, (euler_time, euler_points) in enumerate(clouds_euler.items()):
        euler_ax = euler_axes[euler_row, euler_col]
        euler_ax.scatter(
            euler_points[:, euler_i], euler_points[:, euler_j],
            s=3, alpha=0.35, color='#b45309', linewidths=0, rasterized=True,
        )
        euler_ax.set(
            xlabel=fr'$x_{euler_i + 1}$', ylabel=fr'$x_{euler_j + 1}$',
            xlim=(euler_lo - euler_padding, euler_hi + euler_padding),
            ylim=(euler_lo - euler_padding, euler_hi + euler_padding),
        )
        euler_ax.set_aspect('equal', adjustable='box')
        if euler_row == 0:
            euler_ax.set_title(f't = {euler_time:g}')
euler_fig.suptitle(
    fr'10D two-block explicit Euler: $b={COURNOT_B:g}$, '
    fr'$\mu_A={COURNOT_MU_A:g}$, $\mu_B={COURNOT_MU_B:g}$'
)
euler_fig.savefig(output_dir / 'euler_point_clouds.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from pathlib import Path
import shutil

run_folder = Path(output_dir)
zip_path = shutil.make_archive(
    str(output_root / run_folder.name),
    'zip',
    root_dir=str(run_folder.parent),
    base_dir=run_folder.name,
)

try:
    from google.colab import files
except ImportError:
    print('Saved result archive:', zip_path)
else:
    files.download(zip_path)